# Task 2C — Base vs fine-tuned evaluation, plus the RAG fallback bonus (Colab T4)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nazifmhd/Task-1---Financial-AI/blob/main/task2_genai/03_evaluate.ipynb)

**Fairness:** both models get the **same system prompt**, the **same held-out test set** (never seen in
training), the same 4-bit NF4 loading, and greedy decoding with the same stop tokens. The only variable is fine-tuning.
The fine-tuned model is loaded **from the public Hugging Face repo**, which also proves the published link works.

In [ ]:
# Colab: clone the repo (data + src) and install pinned dependencies.
import os, sys, subprocess
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not os.path.exists("Task-1---Financial-AI"):
        subprocess.run(["git", "clone", "-q", "https://github.com/nazifmhd/Task-1---Financial-AI.git"], check=True)
    os.chdir("Task-1---Financial-AI/task2_genai")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) != "task2_genai" and os.path.isdir("task2_genai"):
    os.chdir("task2_genai")
sys.path.insert(0, os.getcwd())
print("cwd:", os.getcwd(), "| Colab:", IN_COLAB)

In [ ]:
import json, logging, warnings, time
warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s", force=True)
for n in ("httpx", "urllib3", "sentence_transformers", "chromadb", "huggingface_hub", "datasets"):
    logging.getLogger(n).setLevel(logging.WARNING)
import torch, transformers, peft, bitsandbytes, pandas as pd
from IPython.display import display, Markdown, Image
pd.set_option("display.max_colwidth", 140); pd.set_option("display.width", 220)
import config
assert torch.cuda.is_available(), "Runtime -> Change runtime type -> T4 GPU"
p = torch.cuda.get_device_properties(0)
print(f"GPU: {p.name} | {p.total_memory/2**30:.1f} GiB | compute capability {p.major}.{p.minor}")
print("torch", torch.__version__, "| transformers", transformers.__version__, "| peft", peft.__version__,
      "| bitsandbytes", bitsandbytes.__version__)

In [ ]:
# HF token from Colab Secrets (key icon) - never printed, never committed.
from src.llm import read_secret
HF_TOKEN = read_secret("HF_TOKEN")
print("HF token loaded:", bool(HF_TOKEN), "| target repo:", config.HF_REPO_ID)

In [ ]:
from transformers import AutoModelForCausalLM
from src import train as T, evaluate as E
from src.format_split import load_split
test, val, train_rows = load_split("test"), load_split("val"), load_split("train")
print({"train": len(train_rows), "val": len(val), "test": len(test)})

def load_eval_model(name):
    m = AutoModelForCausalLM.from_pretrained(name, quantization_config=T.bnb_config(), device_map={"": 0},
                                             torch_dtype=torch.float16, attn_implementation="sdpa")
    m.eval(); return m, T.load_tokenizer(name)

## 1. Baseline: untuned Phi-3-mini with the same system prompt

In [ ]:
print("System prompt used for BOTH models:\n" + E.STUDENT_SYSTEM_PROMPT + "\n")
base, tok_b = load_eval_model(config.BASE_MODEL)
t0 = time.time(); preds_base = E.predict_rows(base, tok_b, test); print(f"base: {time.time()-t0:.0f}s")
del base; T.free_gpu()

## 2. Fine-tuned (merged) model from the Hub

In [ ]:
tuned, tok_t = load_eval_model(config.HF_REPO_ID)
t0 = time.time(); preds_tuned = E.predict_rows(tuned, tok_t, test); print(f"tuned: {time.time()-t0:.0f}s")
preds_val_tuned = E.predict_rows(tuned, tok_t, val)          # used only to calibrate the RAG threshold

## 3. ROUGE-L, BERTScore and structured metrics

In [ ]:
sb, st = E.score_predictions(test, preds_base), E.score_predictions(test, preds_tuned)
labels = {"rougeL": "ROUGE-L (F)", "bertscore_f1": "BERTScore F1", "json_parse": "JSON parse rate",
          "schema_valid": "Schema-valid rate", "clause_type_acc": "clause_type accuracy",
          "risk_flag_acc": "risk_flag accuracy", "party_match": "party exact match",
          "trigger_null_agree": "trigger null/non-null agreement", "party_in_text": "party grounded in clause",
          "obligation_rougeL": "obligation-field ROUGE-L"}
rows_ = []
for k, name in labels.items():
    a = [float(d[k]) for d in sb["per_example"]]; b = [float(d[k]) for d in st["per_example"]]
    mean, lo, hi = E.bootstrap_diff_ci(a, b)
    rows_.append({"metric": name, "base": sb["aggregate"][k], "fine-tuned": st["aggregate"][k],
                  "Δ": mean, "95% CI of Δ (paired bootstrap)": f"[{lo:+.3f}, {hi:+.3f}]"})
results = pd.DataFrame(rows_)
display(Markdown(f"**Held-out test set: n = {len(test)}**"))
display(results.style.format({"base": "{:.3f}", "fine-tuned": "{:.3f}", "Δ": "{:+.3f}"}).hide(axis="index"))

In [ ]:
print("| Model | ROUGE-L | BERTScore F1 |\n|---|---|---|")
print(f"| Phi-3-mini base (system prompt, no fine-tuning) | {sb['aggregate']['rougeL']:.3f} | {sb['aggregate']['bertscore_f1']:.3f} |")
print(f"| Phi-3-mini QLoRA fine-tuned | {st['aggregate']['rougeL']:.3f} | {st['aggregate']['bertscore_f1']:.3f} |")

## 4. Side-by-side outputs

In [ ]:
side = pd.DataFrame({"id": [r["id"] for r in test],
                     "clause": [r["messages"][1]["content"][:150] for r in test],
                     "base": [p["raw"][:220] for p in preds_base],
                     "fine-tuned": [p["raw"][:220] for p in preds_tuned],
                     "reference": [r["messages"][2]["content"][:220] for r in test]})
display(side.head(8))

## 5. Bonus — RAG fallback for low-confidence answers
* **Confidence** is the mean log-probability of the generated tokens *given the prompt* (from generation scores).
* **Threshold** is the 20th percentile of that confidence on the **validation** set, so roughly the least-confident fifth triggers retrieval. Any schema-invalid answer also triggers it.
* **Knowledge base:** ChromaDB with MiniLM embeddings over the **training** clauses and their gold extractions, plus label definitions and the risk rubric. There is no val/test leakage.

In [ ]:
from src.rag_fallback import ComplianceKB, calibrate_threshold, answer_with_fallback
kb = ComplianceKB(train_rows)
thr = calibrate_threshold([p["mean_logprob"] for p in preds_val_tuned])
print(f"KB documents: {kb.col.count()} | confidence threshold (val p{config.RAG_CONFIDENCE_PERCENTILE}): {thr:.4f}")
rag = [answer_with_fallback(tuned, tok_t, r["messages"][1]["content"], kb, thr, first=p)
       for r, p in zip(test, preds_tuned)]
preds_rag = [{"id": r["id"], **x["final"]} for r, x in zip(test, rag)]
s_rag = E.score_predictions(test, preds_rag, with_bertscore=False)
n_fb = sum(x["used_rag"] for x in rag)
print(f"Fallback triggered on {n_fb}/{len(test)} test clauses")
cmp = pd.DataFrame([{"pipeline": "fine-tuned", **{labels[k]: st["aggregate"][k] for k in ("rougeL", "schema_valid", "clause_type_acc", "risk_flag_acc", "party_match")}},
                    {"pipeline": "fine-tuned + RAG fallback", **{labels[k]: s_rag["aggregate"][k] for k in ("rougeL", "schema_valid", "clause_type_acc", "risk_flag_acc", "party_match")}}])
display(cmp.round(3))

In [ ]:
# Concrete before/after: the fallback case with the largest gain (ROUGE-L + label correctness).
per_t = {d["id"]: d for d in st["per_example"]}; per_r = {d["id"]: d for d in s_rag["per_example"]}
def gain(i): return (per_r[i]["rougeL"] + per_r[i]["clause_type_acc"] + per_r[i]["risk_flag_acc"]) - \
                    (per_t[i]["rougeL"] + per_t[i]["clause_type_acc"] + per_t[i]["risk_flag_acc"])
fb = [(r, x) for r, x in zip(test, rag) if x["used_rag"]]
if fb:
    r, x = max(fb, key=lambda rx: gain(rx[0]["id"]))
    print(f"[{r['id']}] confidence {x['first']['mean_logprob']:.4f} < threshold {thr:.4f} -> retrieved {x['retrieved']}")
    print("\nCLAUSE:\n" + r["messages"][1]["content"])
    print("\nBEFORE (fine-tuned only):\n" + x["first"]["raw"])
    print("\nAFTER (with retrieved context):\n" + x["final"]["raw"])
    print("\nREFERENCE:\n" + r["messages"][2]["content"])
    print(f"\nROUGE-L {per_t[r['id']]['rougeL']:.3f} -> {per_r[r['id']]['rougeL']:.3f} | "
          f"clause_type {per_t[r['id']]['clause_type_acc']:.0f} -> {per_r[r['id']]['clause_type_acc']:.0f} | "
          f"risk {per_t[r['id']]['risk_flag_acc']:.0f} -> {per_r[r['id']]['risk_flag_acc']:.0f}")

## 6. Save artifacts (used by notebook 04) and publish them next to the model

In [ ]:
config.OUTPUT_DIR.mkdir(exist_ok=True)
def dump(name, rows):
    with open(config.OUTPUT_DIR / name, "w", encoding="utf-8") as f:
        for r in rows: f.write(json.dumps(r, ensure_ascii=False) + "\n")
dump("predictions_base.jsonl", preds_base); dump("predictions_tuned.jsonl", preds_tuned)
dump("rag_results.jsonl", [{"id": r["id"], **x} for r, x in zip(test, rag)])
json.dump({"n_test": len(test), "base": sb, "tuned": st, "tuned_rag": s_rag, "rag_threshold": thr,
           "rag_triggered": n_fb}, open(config.OUTPUT_DIR / "metrics.json", "w"), indent=1)
results.to_csv(config.OUTPUT_DIR / "metrics_table.csv", index=False)
if HF_TOKEN:
    from huggingface_hub import HfApi
    api = HfApi(token=HF_TOKEN)
    for f in ("predictions_base.jsonl", "predictions_tuned.jsonl", "rag_results.jsonl", "metrics.json", "metrics_table.csv"):
        api.upload_file(path_or_fileobj=str(config.OUTPUT_DIR / f), path_in_repo=f"eval/{f}", repo_id=config.HF_REPO_ID)
    print("Uploaded eval artifacts to", f"https://huggingface.co/{config.HF_REPO_ID}/tree/main/eval")